# Kaggle Playground S6E8 — Improved CatBoost
**Goal:** improve the confirmed baseline Kaggle ROC-AUC of **0.95801**.

This notebook is intentionally simple and end-to-end:
1. Load data
2. Prepare features
3. Create a small set of behavioral features
4. Run 5-fold Stratified CV
5. Train a stronger CatBoost model
6. Generate `submission_improved.csv`

Keep the old `submission.csv` as the benchmark. Only keep this version if its Kaggle score beats **0.95801**.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

# Reproducibility
SEED = 42
N_FOLDS = 5

# Load
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train:", train.shape)
print("Test :", test.shape)


Train: (691369, 14)
Test : (296302, 13)


In [2]:
# Separate target and ID
y = train["addicted_label"].copy()

X = train.drop(columns=["addicted_label", "id"]).copy()
X_test = test.drop(columns=["id"]).copy()

cat_cols = ["gender", "stress_level", "academic_work_impact"]

# Handle missing categorical values BEFORE making CV splits
for col in cat_cols:
    X[col] = X[col].fillna("Missing").astype(str)
    X_test[col] = X_test[col].fillna("Missing").astype(str)

print("Categorical columns:", cat_cols)
print("Remaining categorical NaNs:")
print(X[cat_cols].isna().sum())


Categorical columns: ['gender', 'stress_level', 'academic_work_impact']
Remaining categorical NaNs:
gender                  0
stress_level            0
academic_work_impact    0
dtype: int64


In [3]:
# -----------------------------
# Controlled feature engineering
# -----------------------------
# These features describe how screen time is distributed.
# Small eps prevents division by zero.

eps = 1e-6

def add_features(df):
    df = df.copy()

    df["leisure_screen_hours"] = (
        df["social_media_hours"].fillna(0)
        + df["gaming_hours"].fillna(0)
    )

    df["screen_minus_work"] = (
        df["daily_screen_time_hours"]
        - df["work_study_hours"]
    )

    df["social_media_share"] = (
        df["social_media_hours"] / (df["daily_screen_time_hours"] + eps)
    )

    df["gaming_share"] = (
        df["gaming_hours"] / (df["daily_screen_time_hours"] + eps)
    )

    df["weekend_to_daily_screen"] = (
        df["weekend_screen_time"]
        / (df["daily_screen_time_hours"] + eps)
    )

    df["notifications_per_app_open"] = (
        df["notifications_per_day"]
        / (df["app_opens_per_day"] + eps)
    )

    return df

X = add_features(X)
X_test = add_features(X_test)

print("New X shape:", X.shape)


New X shape: (691369, 18)


In [4]:
# -----------------------------
# 5-Fold Stratified CV
# -----------------------------
skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

oof_pred = np.zeros(len(X))
test_pred = np.zeros(len(X_test))
fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):

    print("\n" + "=" * 55)
    print(f"FOLD {fold}/{N_FOLDS}")
    print("=" * 55)

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=1800,
        learning_rate=0.035,
        depth=8,
        loss_function="Logloss",
        eval_metric="AUC",
        cat_features=cat_cols,
        verbose=300,
        random_seed=SEED + fold,
        thread_count=-1,
        l2_leaf_reg=5,
        random_strength=0.5
    )

    model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, y_valid),
        early_stopping_rounds=150
    )

    valid_pred = model.predict_proba(X_valid)[:, 1]
    fold_test_pred = model.predict_proba(X_test)[:, 1]

    oof_pred[valid_idx] = valid_pred
    test_pred += fold_test_pred / N_FOLDS

    auc = roc_auc_score(y_valid, valid_pred)
    fold_scores.append(auc)

    print(f"Fold {fold} ROC-AUC: {auc:.6f}")

overall_auc = roc_auc_score(y, oof_pred)

print("\n" + "=" * 55)
print("IMPROVED MODEL RESULTS")
print("=" * 55)

for i, score in enumerate(fold_scores, 1):
    print(f"Fold {i}: {score:.6f}")

print(f"Mean Fold ROC-AUC : {np.mean(fold_scores):.6f}")
print(f"Overall OOF ROC-AUC: {overall_auc:.6f}")



FOLD 1/5
0:	test: 0.9150101	best: 0.9150101 (0)	total: 757ms	remaining: 22m 41s
300:	test: 0.9469077	best: 0.9469077 (300)	total: 2m 25s	remaining: 12m 3s
600:	test: 0.9533575	best: 0.9533575 (600)	total: 4m 45s	remaining: 9m 29s
900:	test: 0.9566091	best: 0.9566091 (900)	total: 7m 1s	remaining: 7m
1200:	test: 0.9584814	best: 0.9584814 (1200)	total: 9m 18s	remaining: 4m 38s
1500:	test: 0.9596512	best: 0.9596512 (1500)	total: 11m 36s	remaining: 2m 18s
1799:	test: 0.9604425	best: 0.9604425 (1799)	total: 13m 57s	remaining: 0us

bestTest = 0.9604425118
bestIteration = 1799

Fold 1 ROC-AUC: 0.960443

FOLD 2/5
0:	test: 0.9136697	best: 0.9136697 (0)	total: 466ms	remaining: 13m 58s
300:	test: 0.9471313	best: 0.9471313 (300)	total: 2m 18s	remaining: 11m 31s
600:	test: 0.9542626	best: 0.9542626 (600)	total: 4m 28s	remaining: 8m 55s
900:	test: 0.9574715	best: 0.9574715 (900)	total: 7m 1s	remaining: 7m
1200:	test: 0.9592629	best: 0.9592629 (1200)	total: 9m 26s	remaining: 4m 42s
1500:	test: 0.9604

In [5]:
# Save CV results
cv_results = pd.DataFrame({
    "fold": range(1, N_FOLDS + 1),
    "roc_auc": fold_scores
})

cv_results.loc[len(cv_results)] = [
    "mean",
    np.mean(fold_scores)
]

cv_results.to_csv("cv_results_improved.csv", index=False)

# Save OOF predictions
oof_df = pd.DataFrame({
    "id": train["id"],
    "actual": y,
    "oof_prediction": oof_pred
})

oof_df.to_csv("oof_predictions_improved.csv", index=False)

print("Saved CV and OOF results.")


Saved CV and OOF results.


In [6]:
# -----------------------------
# Create Kaggle submission
# -----------------------------
submission = pd.DataFrame({
    "id": test["id"],
    "addicted_label": test_pred
})

submission.to_csv("submission_improved.csv", index=False)

print("Saved: submission_improved.csv")
print(submission.head())
print("\nPrediction range:",
      submission["addicted_label"].min(),
      "to",
      submission["addicted_label"].max())


Saved: submission_improved.csv
       id  addicted_label
0  691369        0.999396
1  691370        0.938922
2  691371        0.968135
3  691372        0.981759
4  691373        0.996874

Prediction range: 0.00036046708804675365 to 0.9999989322202233


## Decision rule

Your confirmed benchmark is **0.95801**.

After uploading `submission_improved.csv` to Kaggle:

- **> 0.95801:** keep the improved model.
- **≤ 0.95801:** keep the original `submission.csv`; the improvement did not help.

Do not overwrite the original submission until the new Kaggle score is confirmed.